# Hamilton PREP: Concise demo — teaching needle, liquid transfer, plate movement

Single notebook demonstrating:
1. **Teaching needle** — Pick up teaching tip, move above plate A1 at safe height, drop tip.
2. **Liquid handling** — Tip pickup, dual-channel aspirate and dispense.
3. **Plate movement** — CORE gripper: pick plate from deck[0], drop at deck[1].

**Deck layout:** 1× 50 µL NTR tips at deck[3], 1× plate at deck[0] (moved to deck[1]). Visualizer runs after deck creation.

## 1. Imports and config

In [1]:
import sys
import logging
from asyncio import sleep

from pylabrobot.liquid_handling.backends.hamilton.prep_backend import PrepBackend
from pylabrobot.liquid_handling.backends.hamilton.tcp_backend import HamiltonTCPClient
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.resources import Coordinate
from pylabrobot.resources.hamilton import PrepDeck
from pylabrobot.resources import hamilton_96_tiprack_50uL_NTR, Cor_Axy_96_wellplate_500uL_Ub
from pylabrobot.visualizer import Visualizer

logging.getLogger("pylabrobot").setLevel(logging.INFO)
logging.getLogger("pylabrobot").handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
logging.getLogger("pylabrobot").addHandler(handler)

HOST = "192.168.100.102"
PORT = 2000
SAFE_HEIGHT_MM_ABOVE_WELL = 25

## 2. Deck layout and visualizer

In [2]:
# PrepDeck: spots 0–7 (column-major). With CORE grippers for plate movement.
deck = PrepDeck(with_core_grippers=True)

tip_rack = deck[3] = hamilton_96_tiprack_50uL_NTR(name="ntr_50", with_tips=True)
plate = deck[0] = Cor_Axy_96_wellplate_500uL_Ub("plate")

visualizer = Visualizer(deck, open_browser=False)
await visualizer.setup()

Websocket server started at http://127.0.0.1:2122
File server started at http://127.0.0.1:1338 . Open this URL in your browser.


## 3. Backend and liquid handler setup

In [3]:
backend = PrepBackend(host=HOST, port=PORT)
lh = LiquidHandler(backend=backend, deck=deck)
await lh.setup(smart=True, force_initialize=False)

INFO - Connection initialized (Client ID: 35, Address: 2:35:65535)
INFO - Client registered.
INFO - Setup complete. Registered as Client ID 35 (2:35:65535), Root: MLPrepRoot
WARNING - Unknown introspection type category for type_id=113; treating as parameter
INFO - Interfaces: coordinator, deck_config, mlprep, mlprep_service, mph, pipettor
INFO - MLPrep already initialized, skipping Initialize
INFO - Hardware config: has_enclosure=True, safe_speeds=False, traverse_height=167.5, deck_bounds=DeckBounds(min_x=0.0, max_x=299.0, min_y=-9.0, max_y=385.0, min_z=19.5, max_z=167.5), deck_sites=6, waste_sites=3, num_channels=2, has_mph=True


In [ ]:
# Deck, waste, and other config data from setup can be accessed here.
config = lh.backend._config
print("Instrument configuration:")
print(f"  Deck bounds: {config.deck_bounds}")
print(f"  Has enclosure: {config.has_enclosure}")
print(f"  Safe speeds enabled: {config.safe_speeds_enabled}")
print(f"  Default traverse height: {config.default_traverse_height}")
print(f"  Number of channels: {config.num_channels}")
print(f"  Has MPH: {config.has_mph}")

print("\nDeck sites:")
for site in config.deck_sites:
    print(site)

print("\nWaste sites:")
for waste in config.waste_sites:
    print(waste)

Instrument configuration:
  Deck bounds: DeckBounds(min_x=0.0, max_x=299.0, min_y=-9.0, max_y=385.0, min_z=19.5, max_z=167.5)
  Has enclosure: True
  Safe speeds enabled: False
  Default traverse height: 167.5
  Number of channels: 2
  Has MPH: True

Deck sites:
DeckSiteInfo(id=0, left_bottom_front_x=-2.2300000190734863, left_bottom_front_y=390.0, left_bottom_front_z=0.0, length=310.0, width=100.0, height=167.0)
DeckSiteInfo(id=1, left_bottom_front_x=270.0, left_bottom_front_y=-3.0, left_bottom_front_z=0.0, length=100.0, width=400.0, height=73.0)
DeckSiteInfo(id=2, left_bottom_front_x=284.760009765625, left_bottom_front_y=214.2899932861328, left_bottom_front_z=0.0, length=6.0, width=6.0, height=85.0)
DeckSiteInfo(id=0, left_bottom_front_x=-2.2300000190734863, left_bottom_front_y=390.0, left_bottom_front_z=0.0, length=310.0, width=100.0, height=165.60000610351562)
DeckSiteInfo(id=1, left_bottom_front_x=270.0, left_bottom_front_y=-3.0, left_bottom_front_z=0.0, length=100.0, width=400.0, 

In [5]:
# INTROSPECTION: PREP pipette interface: list command signatures via backend.client.introspect()
PIPETTOR_PATH = "MLPrepRoot.PipettorRoot.Pipettor"
pool, reg = await backend.client.introspect(PIPETTOR_PATH)

print(f"Command signatures on {PIPETTOR_PATH} ({len(reg.methods)} methods):\n")
for m in reg.methods:
    print(f"  {m.get_signature_string(reg)}")

INFO - Global type pool built: 57 structs, 11 enums from 1 global objects
Command signatures on MLPrepRoot.PipettorRoot.Pipettor (49 methods):

  [1:1] Aspirate(aspirateParameters: AspirateParametersNoLldAndMonitoring) -> void
  [1:2] AspirateTadm(aspirateParameters: AspirateParametersNoLldAndTadm) -> void
  [1:3] AspirateLld(aspirateParameters: AspirateParametersLldAndMonitoring) -> void
  [1:4] AspirateLldTadm(aspirateParameters: AspirateParametersLldAndTadm) -> void
  [1:5] Dispense(dispenseParameters: DispenseParametersNoLld) -> void
  [1:6] DispenseLld(dispenseParameters: DispenseParametersLld) -> void
  [1:7] DispenseInitializeToWaste(wasteParameters: DispenseInitToWasteParameters) -> void
  [1:8] PickupTips(tipParameters: TipPositionParameters, finalZ: f32, seekSpeed: f32, tipDefinitionId: u8, tadm: bool, dispenserVolume: f32, dispenserSpeed: f32) -> void
  [1:9] PickupTips(tipParameters: TipPositionParameters, finalZ: f32, seekSpeed: f32, tipDefinition: TipPickupParameters, tad

## 4. Teaching needle: above plate A1 at safe height

In [6]:
# Set Logger to debug so we can see params as sent to the backend.
logging.getLogger("pylabrobot").setLevel(logging.DEBUG)

teaching_tip = deck.get_resource("teaching_tip")
await lh.pick_up_tips([teaching_tip])

a1 = plate.get_item("A1")
safe_pos = a1.get_absolute_location("c", "c", "b") + Coordinate(0, 0, SAFE_HEIGHT_MM_ABOVE_WELL)
await lh.backend.move_to_position(safe_pos.x, safe_pos.y, safe_pos.z)
await sleep(3)

await lh.drop_tips([teaching_tip])

DEBUG - pick_up_tips(tip_spots=['teaching_tip'], use_channels=None, offsets=None)
DEBUG - PrepPickUpTips parameters: {'tip_positions': [TipPositionParameters(default_values=False, channel=<ChannelIndex.RearChannel: 2>, x_position=287.76, y_position=217.29, z_position=75.75, z_seek=88.75)], 'final_z': 167.5, 'seek_speed': 15.0, 'tip_definition': TipPickupParameters(default_values=False, volume=360, length=51.9, tip_type=<TipTypes.StandardVolume: 2>, has_filter=True, is_needle=False, is_tool=False), 'enable_tadm': False, 'dispenser_volume': 0.0, 'dispenser_speed': 250.0}
DEBUG - PrepMoveToPosition parameters: {'move_parameters': GantryMoveXYZParameters(default_values=False, gantry_x_position=13.6, axis_parameters=[ChannelYZMoveParameters(default_values=False, channel=<ChannelIndex.RearChannel: 2>, y_position=75.5, z_position=29.95)])}
DEBUG - drop_tips(tip_spots=['teaching_tip'], use_channels=None, offsets=None, allow_nonzero_volume=False)
DEBUG - PrepDropTips parameters: {'tip_positions

## 5. Tip pickup, aspirate, dispense (dual channel)

In [7]:
tip_spots = tip_rack['A1:B1']
await lh.pick_up_tips(tip_spots)

await lh.aspirate(
    plate["A1:B1"],
    vols=[35, 25],
    liquid_height=[3, 3],
    z_liquid_exit_speed = [25, 25],
)

await lh.dispense(
    plate['A7:B7'],
    vols=[35, 25],
    liquid_height=[3, 3],
    z_liquid_exit_speed = [25, 25],
)

#await lh.drop_tips(tip_spots)
await lh.discard_tips()

DEBUG - pick_up_tips(tip_spots=['ntr_50_tipspot_A1', 'ntr_50_tipspot_B1'], use_channels=None, offsets=None)
DEBUG - PrepPickUpTips parameters: {'tip_positions': [TipPositionParameters(default_values=False, channel=<ChannelIndex.RearChannel: 2>, x_position=13.525, y_position=361.5, z_position=59.650000000000006, z_seek=72.65), TipPositionParameters(default_values=False, channel=<ChannelIndex.FrontChannel: 1>, x_position=13.525, y_position=352.5, z_position=59.650000000000006, z_seek=72.65)], 'final_z': 167.5, 'seek_speed': 15.0, 'tip_definition': TipPickupParameters(default_values=False, volume=65, length=42.4, tip_type=<TipTypes.StandardVolume: 2>, has_filter=False, is_needle=False, is_tool=False), 'enable_tadm': False, 'dispenser_volume': 0.0, 'dispenser_speed': 250.0}
DEBUG - aspirate(resources=['plate_well_A1', 'plate_well_B1'], vols=[35, 25], use_channels=None, flow_rates=None, offsets=None, liquid_height=[3, 3], blow_out_air_volume=None)
DEBUG - PrepAspirateNoLldMonitoringV2 param

## 6. Plate movement with CORE gripper (deck[0] → deck[1])

In [8]:
await lh.pick_up_resource(plate)
await lh.drop_resource(destination=deck[1], return_gripper=True)

DEBUG - pick_up_resource(resource=plate, offset=Coordinate(000.000, 000.000, 000.000), pickup_distance_from_top=None, direction=GripDirection.FRONT)
DEBUG - No preferred pickup location for resource plate. Using default pickup distance of 5mm.
DEBUG - PrepPickUpTool parameters: {'tip_definition': TipPickupParameters(default_values=False, volume=1.0, length=22.9, tip_type=<TipTypes.None_: 0>, has_filter=False, is_needle=False, is_tool=True), 'tool_position_x': 290.0, 'tool_position_z': 62.5, 'front_channel_position_y': 257.5, 'rear_channel_position_y': 275.5, 'tool_seek': 72.5, 'tool_x_radius': 2.0, 'tool_y_radius': 2.0}
DEBUG - PrepMoveZUpToSafe parameters: {'channels': [<ChannelIndex.RearChannel: 2>, <ChannelIndex.FrontChannel: 1>]}
DEBUG - PrepPickUpPlate parameters: {'plate_top_center': XYZCoord(default_values=False, x_position=63.5, y_position=44.255, z_position=18.57), 'plate': PlateDimensions(default_values=False, length=127.0, width=85.51, height=14.82), 'clearance_y': 2.5, 'gri

## 7. Teardown

In [10]:
logging.getLogger("pylabrobot").setLevel(logging.INFO)

await lh.backend.park()
await lh.backend.disco_mode()
await lh.stop()

INFO - Closing connection to socket 192.168.100.102:2000
INFO - Hamilton TCP client stopped
